<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Belgium_Gdp_and_consumption_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
BELGIUM: Consumption, GDP and the long run  -  full empirical pipeline
=========================================================================
Downloads data (AMECO via DBnomics, or World Bank), builds the variables and runs:

  1. descriptives, linear trend and exponential growth
  2. unit roots: ADF, KPSS, Zivot-Andrews (structural break), order of integration of every variable
  3. Engle-Granger (both directions), DOLS long-run elasticity, robustness with real disposable income
  4. Error-correction model (with crisis dummies), half-life, diagnostics
  5. Johansen + VECM  (2-variable and, if data exist, 3-variable system with investment)
  6. Granger causality (differences) and Toda-Yamamoto (levels)
  7. ARDL bounds test with extra variables (real long-term interest rate, unemployment)
  8. Impulse responses and variance decomposition
  9. Auto-generated summary of findings

Usage
-----
  pip install pandas numpy statsmodels matplotlib requests openpyxl
  python belgium_project_full.py                       # AMECO (1960-), needs internet
  python belgium_project_full.py --source wb           # World Bank (1970-): gdp, cons, inv only
  python belgium_project_full.py --csv my_data.csv     # offline; columns listed below

CSV columns (lower-case; only year, gdp, cons are compulsory; all real/constant prices):
  year, gdp, cons, inv, gov, unemp, ltrate, defl, hhdi_nom
    gdp      GDP at constant prices                       (AMECO OVGD)
    cons     Private final consumption, constant prices   (AMECO OCPH)
    inv      Gross fixed capital formation, constant      (AMECO OIGT)
    gov      Government final consumption, constant       (AMECO OCTG)
    unemp    Unemployment rate, %                         (AMECO ZUTN)
    ltrate   Nominal long-term interest rate, %           (AMECO ILN)
    defl     Private consumption deflator, 2015=100       (AMECO PCPH)
    hhdi_nom Gross disposable income of households, nominal  (AMECO UVGH)
"""
import argparse
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.regression.recursive_ls import RecursiveLS
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan, breaks_cusumolsresid
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.tsa.api import VAR
from statsmodels.tsa.ardl import UECM, ardl_select_order
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests, kpss, zivot_andrews
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_coint_rank

warnings.filterwarnings("ignore")

OUT = "results_belgium"
REPORT, TABLES, SUMMARY = [], {}, {}

AMECO_CODES = {  # our name -> AMECO variable code
    "gdp": "OVGD",
    "cons": "OCPH",
    "inv": "OIGT",
    "gov": "OCTG",
    "unemp": "ZUTN",
    "ltrate": "ILN",
    "defl": "PCPH",
    "hhdi_nom": "UVGH",
}


# ======================================================================================
# small utilities
# ======================================================================================
def P(*args):
    line = " ".join(str(a) for a in args)
    print(line)
    REPORT.append(line)


def H(title):
    P("\n" + "=" * 90 + f"\n{title}\n" + "=" * 90)


def save_table(name, df):
    TABLES[name] = df
    df.to_csv(f"{OUT}/table_{name}.csv", index=False)


def step(fn):
    """run one analysis block; a failure in one block never kills the rest"""

    def wrapper(*a, **k):
        try:
            return fn(*a, **k)
        except Exception as e:
            P(f"\n!! Step '{fn.__name__}' failed: {type(e).__name__}: {e}")
            return None

    wrapper.__name__ = fn.__name__
    return wrapper


# ======================================================================================
# 0. DATA DOWNLOAD
# ======================================================================================
def fetch_ameco(var):
    """AMECO series for Belgium through the free DBnomics API (national currency / level series)."""
    import requests

    url = f"https://api.db.nomics.world/v22/series/AMECO/{var}"
    params = {"dimensions": json.dumps({"geo": ["bel"]}), "observations": 1, "limit": 50}
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    docs = r.json()["series"]["docs"]
    cands = []
    for d in docs:
        parts = d["series_code"].split(".")
        if parts[0] == "BEL" and len(parts) >= 4 and parts[3] == "0":  # national-currency / index version
            cands.append(d)
    if not cands:
        raise ValueError(f"no Belgian national-currency series found for {var}")
    d = cands[0]
    s = pd.Series(pd.to_numeric(pd.Series(d["value"]), errors="coerce").values,
                  index=[int(str(p)[:4]) for p in d["period"]])
    return s.dropna().sort_index()


def fetch_wb(code, start=1960, end=2030):
    import requests

    url = f"https://api.worldbank.org/v2/country/BEL/indicator/{code}"
    r = requests.get(url, params={"format": "json", "per_page": 500, "date": f"{start}:{end}"}, timeout=60)
    r.raise_for_status()
    rows = r.json()[1]
    return pd.Series({int(x["date"]): x["value"] for x in rows if x["value"] is not None}).sort_index()


def load_data(source, csv, start, end):
    if csv:
        df = pd.read_csv(csv) if csv.endswith(".csv") else pd.read_excel(csv)
        df.columns = [c.strip().lower() for c in df.columns]
        df = df.set_index("year").apply(pd.to_numeric, errors="coerce")
    elif source == "wb":
        wb = {"gdp": "NY.GDP.MKTP.KN", "cons": "NE.CON.PRVT.KN", "inv": "NE.GDI.FTOT.KN"}
        cols = {}
        for k, code in wb.items():
            try:
                cols[k] = fetch_wb(code)
                P(f"  downloaded {k:9s} <- World Bank {code}  ({cols[k].index.min()}-{cols[k].index.max()})")
            except Exception as e:
                P(f"  !! could not download {k} ({code}): {e}")
        df = pd.concat(cols, axis=1)
    else:
        cols = {}
        for k, code in AMECO_CODES.items():
            try:
                cols[k] = fetch_ameco(code)
                P(f"  downloaded {k:9s} <- AMECO {code:5s} ({cols[k].index.min()}-{cols[k].index.max()})")
            except Exception as e:
                P(f"  !! could not download {k} (AMECO {code}): {e}")
        df = pd.concat(cols, axis=1)
    if "gdp" not in df or "cons" not in df:
        raise SystemExit("GDP and consumption are compulsory. Use --csv with your own file.")
    df = df[(df.index >= start) & (df.index <= end)]  # AMECO's last years are EC FORECASTS: cut them off
    df = df.dropna(subset=["gdp", "cons"])
    df.index = pd.DatetimeIndex(pd.to_datetime(df.index.astype(int).astype(str), format="%Y"), freq="YS")
    df.reset_index().rename(columns={"index": "year"}).to_csv(f"{OUT}/data_downloaded.csv", index=False)
    return df


def build_variables(raw):
    d = pd.DataFrame(index=raw.index)
    d["gdp"], d["cons"] = raw["gdp"], raw["cons"]
    d["lgdp"], d["lcons"] = np.log(raw["gdp"]), np.log(raw["cons"])
    d["cons_share"] = raw["cons"] / raw["gdp"]
    if "inv" in raw:
        d["linv"] = np.log(raw["inv"])
    if "gov" in raw:
        d["lgov"] = np.log(raw["gov"])
    if "unemp" in raw:
        d["unemp"] = raw["unemp"]
    if "defl" in raw:
        d["infl"] = 100 * np.log(raw["defl"]).diff()
    if "ltrate" in raw:
        d["rate"] = raw["ltrate"] - d["infl"] if "infl" in d else raw["ltrate"]  # ex-post real rate
    if "hhdi_nom" in raw and "defl" in raw:
        d["lrhdi"] = np.log(raw["hhdi_nom"] / raw["defl"] * 100)  # real disposable income
    d["t"] = np.arange(1, len(d) + 1)
    for v in ["lgdp", "lcons", "linv", "lgov", "lrhdi"]:
        if v in d:
            d["d" + v] = d[v].diff()
    years = d.index.year
    for y in (1975, 2009, 2020):
        if y in years:
            d[f"D{y}"] = (years == y).astype(float)
    return d


# ======================================================================================
# tests
# ======================================================================================
def adf_row(x, reg, label, maxlag=4):
    x = x.dropna()
    stat, p, lag, nobs, crit, _ = adfuller(x, regression=reg, autolag="BIC", maxlag=maxlag)
    return dict(series=label, deterministics={"c": "const", "ct": "const+trend", "n": "none"}[reg],
                lags=lag, nobs=nobs, t_stat=round(stat, 3), crit_5pct=round(crit["5%"], 3),
                p_value=round(p, 4), decision="Reject H0 -> stationary" if p < 0.05 else "Unit root not rejected")


def integration_order(x, reg):
    x = x.dropna()
    if adfuller(x, regression=reg, autolag="BIC", maxlag=4)[1] < 0.05:
        return 0
    if adfuller(x.diff().dropna(), regression="c", autolag="BIC", maxlag=4)[1] < 0.05:
        return 1
    if adfuller(x.diff().diff().dropna(), regression="c", autolag="BIC", maxlag=4)[1] < 0.05:
        return 2
    return 3


def toda_yamamoto(y, x, p, dmax=1):
    m = p + dmax
    d = pd.DataFrame({"y": y, "x": x})
    for i in range(1, m + 1):
        d[f"y{i}"], d[f"x{i}"] = d["y"].shift(i), d["x"].shift(i)
    d = d.dropna()
    X = sm.add_constant(d[[f"y{i}" for i in range(1, m + 1)] + [f"x{i}" for i in range(1, m + 1)]])
    res = sm.OLS(d["y"], X).fit()
    w = res.wald_test(", ".join(f"x{i} = 0" for i in range(1, p + 1)), use_f=False, scalar=True)
    return float(w.statistic), float(w.pvalue)


def dols(y, x, leads=1, lags=1):
    """Dynamic OLS (Stock-Watson): long-run coefficient with valid (HAC) standard errors."""
    d = pd.DataFrame({"y": y, "x": x})
    dx = x.diff()
    for j in range(-leads, lags + 1):
        d[f"dx_{j}"] = dx.shift(j)
    d = d.dropna()
    X = sm.add_constant(d.drop(columns="y"))
    return sm.OLS(d["y"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 2})


def johansen_block(data, label, maxlags=3):
    P(f"\n--- Johansen system: {label} ---")
    sel = VAR(data).select_order(maxlags=maxlags)
    P(sel.summary())
    p_aic = max(sel.selected_orders["aic"], 2)
    k = p_aic - 1
    P(f"VAR lag (AIC, minimum 2) = {p_aic}  ->  k_ar_diff = {k};  deterministic case: unrestricted constant")
    jo = coint_johansen(data, det_order=0, k_ar_diff=k)
    n = data.shape[1]
    tab = pd.DataFrame({"H0: rank <=": range(n), "trace": jo.lr1.round(3), "trace_crit_5%": jo.cvt[:, 1].round(3),
                        "maxeig": jo.lr2.round(3), "maxeig_crit_5%": jo.cvm[:, 1].round(3)})
    P(tab.to_string(index=False))
    save_table("johansen_" + label.replace(" ", "_").replace(",", ""), tab)
    rank = select_coint_rank(data, det_order=0, k_ar_diff=k, method="trace", signif=0.05).rank
    P(f">>> Cointegrating rank (trace test, 5%) = {rank}")
    if rank >= 1:
        v = VECM(data, k_ar_diff=k, coint_rank=rank, deterministic="co").fit()
        P("beta (cointegrating vectors, normalised on first variable):")
        P(pd.DataFrame(v.beta, index=data.columns).round(4).to_string())
        P("alpha (adjustment/loading coefficients, rows = equations):")
        P(pd.DataFrame(v.alpha, index=data.columns).round(4).to_string())
    return rank, p_aic


# ======================================================================================
# ANALYSIS STEPS
# ======================================================================================
@step
def s1_descriptives(d):
    H("1. DESCRIPTIVES, LINEAR TREND AND EXPONENTIAL GROWTH")
    P(f"Sample {d.index[0].year}-{d.index[-1].year}  (N = {len(d)}).  Variables available: "
      f"{[c for c in ['lgdp','lcons','linv','lrhdi','rate','unemp','infl'] if c in d]}")
    P(f"Average consumption/GDP ratio = {d.cons_share.mean():.3f}  (first year {d.cons_share.iloc[0]:.3f}, "
      f"last {d.cons_share.iloc[-1]:.3f})")
    rows = []
    for v in ["gdp", "cons"]:
        lin = sm.OLS(d[v], sm.add_constant(d["t"])).fit()
        ex = sm.OLS(np.log(d[v]), sm.add_constant(d["t"])).fit()
        rows.append(dict(variable=v, linear_R2=round(lin.rsquared, 4), exp_R2=round(ex.rsquared, 4),
                         growth_cont_pct=round(ex.params["t"] * 100, 3),
                         growth_compound_pct=round((np.exp(ex.params["t"]) - 1) * 100, 3)))
    tab = pd.DataFrame(rows)
    P(tab.to_string(index=False))
    save_table("trend_growth", tab)
    SUMMARY["growth_gdp"] = rows[0]["growth_compound_pct"]
    SUMMARY["growth_cons"] = rows[1]["growth_compound_pct"]

    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    d[["lgdp", "lcons"]].plot(ax=ax[0, 0], title="Log real GDP and private consumption")
    d[["dlgdp", "dlcons"]].plot(ax=ax[0, 1], title="Annual growth (log differences)")
    d["cons_share"].plot(ax=ax[1, 0], title="Consumption share of GDP")
    extra = [c for c in ["rate", "unemp", "infl"] if c in d]
    if extra:
        d[extra].plot(ax=ax[1, 1], title="Real long-term rate, unemployment, inflation (%)")
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig1_overview.png", dpi=150)
    plt.close()


@step
def s2_unit_roots(d):
    H("2. UNIT ROOT TESTS   (ADF H0: unit root | KPSS H0: stationary | Zivot-Andrews H0: unit root, no break)")
    trending = [v for v in ["lgdp", "lcons", "linv", "lgov", "lrhdi"] if v in d]
    flat = [v for v in ["rate", "unemp", "infl"] if v in d]
    rows = []
    for v in trending:
        rows += [adf_row(d[v], "ct", f"{v} (level)"), adf_row(d[v].diff(), "c", f"D {v}")]
    for v in flat:
        rows += [adf_row(d[v], "c", f"{v} (level)"), adf_row(d[v].diff(), "c", f"D {v}")]
    tab = pd.DataFrame(rows)
    P(tab.to_string(index=False))
    save_table("adf", tab)

    k = []
    for v in trending:
        s, _, _, c = kpss(d[v].dropna(), regression="ct", nlags="auto")
        k.append(dict(series=f"{v} (level)", kpss_stat=round(s, 3), crit_5pct=c["5%"],
                      decision="Reject -> NOT stationary" if s > c["5%"] else "Stationary not rejected"))
    tk = pd.DataFrame(k)
    P("\nKPSS (levels, trend):")
    P(tk.to_string(index=False))
    save_table("kpss", tk)

    P("\nZivot-Andrews (one endogenous structural break, level series, break in const+trend):")
    za_rows = []
    for v in ["lgdp", "lcons"]:
        stat, p, crit, lag, bp = zivot_andrews(d[v].dropna(), maxlag=4, regression="ct", autolag="AIC")
        yr = d[v].dropna().index[int(bp)].year
        za_rows.append(dict(series=v, za_stat=round(stat, 3), crit_5pct=round(crit["5%"], 3), p_value=round(p, 4),
                            break_year=yr, decision="Reject -> stationary with break" if p < 0.05 else "Unit root holds"))
    tz = pd.DataFrame(za_rows)
    P(tz.to_string(index=False))
    save_table("zivot_andrews", tz)

    orders = {v: integration_order(d[v], "ct" if v in trending else "c") for v in trending + flat}
    P("\nOrder of integration:", orders)
    SUMMARY["orders"] = orders
    if any(o >= 2 for o in orders.values()):
        P("!! WARNING: a variable looks I(2). Cointegration/ARDL bounds tests are not valid with I(2) variables.")


@step
def s3_engle_granger(d):
    H("3. LONG-RUN RELATIONSHIP: ENGLE-GRANGER, DOLS, ROBUSTNESS")
    lr = sm.OLS(d["lcons"], sm.add_constant(d["lgdp"])).fit()
    P("Static long-run OLS  lcons = a + b*lgdp  (t-values NOT valid for inference; super-consistent point estimate)")
    P(lr.summary().tables[1])
    beta = lr.params["lgdp"]
    ect = lr.resid.rename("ect")
    d["ect"] = ect
    st, p, crit = coint(d["lcons"], d["lgdp"], trend="c", autolag="bic", maxlag=4)
    P(f"\nEngle-Granger (lcons on lgdp): t = {st:.3f}, MacKinnon p = {p:.4f}; crit 1/5/10% = {np.round(crit, 3)}")
    st2, p2, _ = coint(d["lgdp"], d["lcons"], trend="c", autolag="bic", maxlag=4)
    P(f"Engle-Granger (lgdp on lcons): t = {st2:.3f}, MacKinnon p = {p2:.4f}   (reverse normalisation check)")
    SUMMARY.update(eg_p=p, eg_p_rev=p2, beta_ols=beta)

    dl = dols(d["lcons"], d["lgdp"])
    P("\nDOLS (1 lead, 1 lag of D lgdp, HAC std. errors) - valid inference on the long-run elasticity:")
    P(f"   beta_DOLS = {dl.params['x']:.3f}  (se {dl.bse['x']:.3f});  95% CI [{dl.conf_int().loc['x', 0]:.3f}, "
      f"{dl.conf_int().loc['x', 1]:.3f}]")
    t_one = (dl.params["x"] - 1) / dl.bse["x"]
    P(f"   H0: beta = 1 (proportionality):  t = {t_one:.2f}  ->  {'reject' if abs(t_one) > 2 else 'cannot reject'} at ~5%")
    SUMMARY.update(beta_dols=dl.params["x"], beta_dols_se=dl.bse["x"])

    if "lrhdi" in d and d["lrhdi"].notna().sum() >= 25:
        s = d[["lcons", "lrhdi"]].dropna()
        stj, pj, _ = coint(s["lcons"], s["lrhdi"], trend="c", autolag="bic", maxlag=3)
        b = sm.OLS(s["lcons"], sm.add_constant(s["lrhdi"])).fit().params["lrhdi"]
        P(f"\nROBUSTNESS with real household disposable income ({s.index[0].year}-{s.index[-1].year}, N={len(s)}): "
          f"beta = {b:.3f}, EG p = {pj:.4f}")
        SUMMARY.update(rhdi_beta=b, rhdi_p=pj)

    fig, ax = plt.subplots(figsize=(9, 3.8))
    ect.plot(ax=ax, title="Error-correction term: log C minus its long-run value given log GDP")
    ax.axhline(0, color="k", lw=0.8)
    plt.tight_layout()
    plt.savefig(f"{OUT}/fig2_ect.png", dpi=150)
    plt.close()


@step
def s4_ecm(d):
    H("4. ERROR CORRECTION MODEL")
    dm = pd.DataFrame({"dlcons": d["dlcons"], "dlgdp": d["dlgdp"], "ect_1": d["ect"].shift(1)})
    dums = [c for c in d.columns if c.startswith("D") and c[1:].isdigit()]
    for c in dums:
        dm[c] = d[c]
    dm = dm.dropna()
    X0 = sm.add_constant(dm[["dlgdp", "ect_1"]])
    ecm0 = sm.OLS(dm["dlcons"], X0).fit()
    P("(a) Baseline ECM:")
    P(ecm0.summary().tables[1])
    ecm1 = sm.OLS(dm["dlcons"], sm.add_constant(dm[["dlgdp", "ect_1"] + dums])).fit()
    P(f"\n(b) ECM with crisis dummies {dums}:")
    P(ecm1.summary().tables[1])
    rho = ecm0.params["ect_1"]
    hl = np.log(0.5) / np.log(1 + rho) if -1 < rho < 0 else np.nan
    P(f"\nShort-run elasticity = {ecm0.params['dlgdp']:.3f} ; adjustment speed rho = {rho:.3f} "
      f"(t = {ecm0.tvalues['ect_1']:.2f}) ; half-life = {hl:.2f} years")
    SUMMARY.update(rho=rho, rho_t=ecm0.tvalues["ect_1"], half_life=hl, sr_elast=ecm0.params["dlgdp"],
                   rho_dummies=ecm1.params["ect_1"])

    P("\nDiagnostics (baseline ECM):")
    P(f"  Durbin-Watson = {durbin_watson(ecm0.resid):.3f}")
    P(f"  Breusch-Godfrey (2 lags) p = {acorr_breusch_godfrey(ecm0, nlags=2)[1]:.4f}  (H0 no serial correlation)")
    P(f"  Breusch-Pagan            p = {het_breuschpagan(ecm0.resid, ecm0.model.exog)[1]:.4f}  (H0 homoskedastic)")
    P(f"  Jarque-Bera              p = {jarque_bera(ecm0.resid)[1]:.4f}  (H0 normal)")
    P(f"  CUSUM                    p = {breaks_cusumolsresid(ecm0.resid, ddof=3)[1]:.4f}  (H0 stable parameters)")
    try:
        RecursiveLS(dm["dlcons"], X0).fit().plot_cusum().savefig(f"{OUT}/fig3_cusum.png", dpi=150)
        plt.close("all")
    except Exception as e:
        P("  (CUSUM plot skipped:", e, ")")


@step
def s5_johansen(d):
    H("5. JOHANSEN COINTEGRATION AND VECM")
    r2, p2 = johansen_block(d[["lcons", "lgdp"]].dropna(), "lcons lgdp")
    SUMMARY.update(rank2=r2, var_lag=p2)
    for e in [c for c in ("linv", "lgov") if c in d]:
        r3, _ = johansen_block(d[["lcons", "lgdp", e]].dropna(), f"lcons lgdp {e}")
        SUMMARY[f"rank3_{e}"] = r3


@step
def s6_causality(d):
    H("6. GRANGER CAUSALITY")
    k = max(SUMMARY.get("var_lag", 2) - 1, 1)
    dd = d[["dlcons", "dlgdp"]].dropna()
    P(f"(a) Standard F-tests on DIFFERENCES (valid if not cointegrated), lags 1..{k + 1}")
    out = {}
    for cause, effect in [("dlgdp", "dlcons"), ("dlcons", "dlgdp")]:
        try:
            r = grangercausalitytests(dd[[effect, cause]], maxlag=k + 1, verbose=False)
        except TypeError:
            r = grangercausalitytests(dd[[effect, cause]], maxlag=k + 1)
        pv = {int(l): round(float(r[l][0]["ssr_ftest"][1]), 4) for l in r}
        out[f"{cause}->{effect}"] = pv
        P(f"   {cause} -> {effect}: p-values by lag {pv}")
    p = max(SUMMARY.get("var_lag", 2), 1)
    P(f"\n(b) Toda-Yamamoto on LEVELS (valid whether or not cointegrated): VAR({p}) + 1 extra lag")
    for cause, effect in [("lgdp", "lcons"), ("lcons", "lgdp")]:
        w, pv = toda_yamamoto(d[effect], d[cause], p=p)
        P(f"   {cause} -> {effect}: Wald chi2 = {w:.3f}, p = {pv:.4f}")
        SUMMARY[f"ty_{cause}_{effect}"] = pv


@step
def s7_ardl(d):
    H("7. ARDL BOUNDS TEST WITH EXTRA VARIABLES (Pesaran-Shin-Smith)")
    xs = ["lgdp"] + [c for c in ["lgov", "rate", "unemp"] if c in d]
    s = d[["lcons"] + xs].dropna()
    P(f"Dependent: lcons | regressors: {xs} | sample {s.index[0].year}-{s.index[-1].year} (N={len(s)})")
    orders = SUMMARY.get("orders", {})
    if any(orders.get(v, 0) >= 2 for v in ["lcons"] + xs):
        P("!! At least one variable appears I(2): the bounds test is not valid. Fix that first.")
    sel = ardl_select_order(s["lcons"], 2, s[xs], 2, ic="aic", trend="c")
    P("AIC-selected ARDL orders (p, q_1..q_k):", sel.model.ardl_order)
    P(sel.model.fit().summary().tables[1])
    # the bounds-test (UECM) needs >= 1 lag of every regressor in levels, so raise any zero order to 1
    p_lag = max(list(sel.model.ar_lags) + [1])
    q_ord = {x: max(max(sel.model.dl_lags.get(x, [0])), 1) for x in xs}  # keep ALL regressors, AIC-dropped ones too
    P(f"UECM specification used for the bounds test: p = {p_lag}, q = {q_ord}  (all regressors kept on purpose)")
    u = UECM(s["lcons"], p_lag, s[xs], order=q_ord, trend="c").fit()
    bt = u.bounds_test(case=3)
    P(f"\nBounds F-statistic = {bt.statistic:.3f}   (k = {len(xs)} regressors, case 3: unrestricted constant)")
    P(bt.critical_values.round(3).to_string())
    crit5 = bt.critical_values.loc[95.0] if 95.0 in bt.critical_values.index else None
    verdict = "inconclusive"
    if crit5 is not None:
        verdict = ("level relationship EXISTS" if bt.statistic > crit5["upper"]
                   else "NO level relationship" if bt.statistic < crit5["lower"] else "inconclusive")
    P(f">>> At 5%: {verdict}")
    SUMMARY["ardl_verdict"], SUMMARY["ardl_F"] = verdict, bt.statistic
    pr = u.params
    ycoef = [c for c in pr.index if c.startswith("lcons.L1")][0]
    P(f"Adjustment coefficient (lcons.L1) = {pr[ycoef]:.3f}")
    rows = []
    for x in xs:
        xc = [c for c in pr.index if c.startswith(f"{x}.L1")]
        if xc:
            rows.append(dict(variable=x, long_run_coef=round(-pr[xc[0]] / pr[ycoef], 4)))
    tab = pd.DataFrame(rows)
    P("Long-run coefficients (= -coef(x.L1)/coef(lcons.L1)):")
    P(tab.to_string(index=False))
    save_table("ardl_longrun", tab)


@step
def s8_irf_fevd(d):
    H("8. IMPULSE RESPONSES AND VARIANCE DECOMPOSITION (custom publication-quality plots)")
    import var_plots as vp  # must sit in the same folder as this script

    pretty = {"lgdp": "GDP", "lcons": "Consumption", "linv": "Investment", "lgov": "Gov. consumption",
              "dlgdp": "GDP growth", "dlcons": "Consumption growth", "dlinv": "Investment growth",
              "dlgov": "Gov. consumption growth"}
    jobs = [("2var", ["lgdp", "lcons"], SUMMARY.get("rank2", 0))]
    for e in [c for c in ("linv", "lgov") if c in d]:
        jobs.append((f"3var_{e[1:]}", ["lgdp", e, "lcons"], SUMMARY.get(f"rank3_{e}", 0)))
    for tag, cols, rank in jobs:
        if rank >= 1:
            data, kind = d[cols].dropna(), "levels"
            unit, cum = "%", False
        else:
            data, kind = d[["d" + c for c in cols]].dropna(), "differences"
            unit, cum = "pp", True
        p = max(VAR(data).select_order(maxlags=3).selected_orders["bic"], 1)
        res = VAR(data).fit(p)
        labels = [pretty[c] for c in data.columns]
        order = " \u2192 ".join(labels)
        note = (f"VAR({p}) in {kind} ({'cointegrated, consistent in levels' if kind == 'levels' else 'no cointegration'});"
                f" Cholesky ordering: {order} (first = most exogenous).")
        P(f"\n[{tag}] VAR({p}) in {kind}; ordering {order}")
        tabs = vp.make_all(res, labels, OUT, f"fig_{tag}", steps=10, scale=100.0,
                           unit="%" if kind == "levels" else "pp", cumulative_too=True, ordering_note=note)
        P(tabs["irf_peaks"].to_string(index=False))
        P("\nFEVD: Cholesky vs Generalised (ordering-invariant), % of forecast-error variance")
        P(tabs["fevd_comparison"].to_string(index=False))
        save_table(f"{tag}_irf_peaks", tabs["irf_peaks"])
        save_table(f"{tag}_fevd_comparison", tabs["fevd_comparison"])
        P("Figures:", ", ".join(os.path.basename(f) for f in tabs["files"]["figure"]))


def final_summary():
    H("9. SUMMARY OF FINDINGS (auto-generated: check against the tables above)")
    S = SUMMARY
    g = lambda k, f="{:.3f}": f.format(S[k]) if k in S and pd.notna(S[k]) else "n/a"
    P(f"- Growth: GDP {g('growth_gdp','{:.2f}')}% p.a., consumption {g('growth_cons','{:.2f}')}% p.a.")
    P(f"- Orders of integration: {S.get('orders', 'n/a')}")
    P(f"- Engle-Granger p-values: {g('eg_p','{:.4f}')} (cons on gdp), {g('eg_p_rev','{:.4f}')} (reverse)")
    P(f"- Long-run elasticity: OLS {g('beta_ols')}, DOLS {g('beta_dols')} (se {g('beta_dols_se')})")
    P(f"- ECM: adjustment speed {g('rho')} (t = {g('rho_t','{:.2f}')}), half-life {g('half_life','{:.2f}')} yrs, "
      f"short-run elasticity {g('sr_elast')}")
    P(f"- Johansen rank: 2-variable = {S.get('rank2', 'n/a')}; 3-variable systems: "
      f"{ {k: v for k, v in S.items() if k.startswith('rank3_')} or 'n/a' }")
    P(f"- Toda-Yamamoto p: gdp->cons {g('ty_lgdp_lcons','{:.4f}')}, cons->gdp {g('ty_lcons_lgdp','{:.4f}')}")
    P(f"- ARDL bounds test: F = {g('ardl_F','{:.2f}')} -> {S.get('ardl_verdict', 'n/a')}")


def main(source, csv, start, end, outdir):
    global OUT
    OUT = outdir
    os.makedirs(OUT, exist_ok=True)
    H("0. DATA")
    raw = load_data(source, csv, start, end)
    d = build_variables(raw)
    P(f"\nFinal sample: {d.index[0].year}-{d.index[-1].year}, N = {len(d)}")
    for fn in (s1_descriptives, s2_unit_roots, s3_engle_granger, s4_ecm, s5_johansen, s6_causality, s7_ardl,
               s8_irf_fevd):
        fn(d)
    final_summary()
    d.to_csv(f"{OUT}/data_used.csv")
    with open(f"{OUT}/report.txt", "w") as f:
        f.write("\n".join(REPORT))
    try:
        with pd.ExcelWriter(f"{OUT}/all_tables.xlsx") as xw:
            for k, v in TABLES.items():
                v.to_excel(xw, sheet_name=k[:31], index=False)
    except Exception as e:
        P("(Excel export skipped:", e, ")")
    P(f"\nDone. Everything is in ./{OUT}/  (report.txt, all_tables.xlsx, figures, data_used.csv)")


if __name__ == "__main__":
    # Check if we are running in an interactive Jupyter / Colab environment
    if 'ipykernel' in sys.modules:
        # Default fallback values for interactive notebooks to avoid argparse system exit
        main(source="ameco", csv=None, start=1960, end=2024, outdir="results_belgium")
    else:
        ap = argparse.ArgumentParser()
        ap.add_argument("--source", choices=["ameco", "wb"], default="ameco")
        ap.add_argument("--csv", help="offline CSV/Excel (see docstring for columns)")
        ap.add_argument("--start", type=int, default=1960)
        ap.add_argument("--end", type=int, default=2024, help="last year of ACTUAL data (AMECO ends with forecasts)")
        ap.add_argument("--outdir", default="results_belgium")
        a = ap.parse_args()
        main(a.source, a.csv, a.start, a.end, a.outdir)



0. DATA
  downloaded gdp       <- AMECO OVGD  (1960-2027)
  downloaded cons      <- AMECO OCPH  (1960-2027)
  downloaded inv       <- AMECO OIGT  (1960-2027)
  downloaded gov       <- AMECO OCTG  (1960-2027)
  downloaded unemp     <- AMECO ZUTN  (1960-2027)
  downloaded ltrate    <- AMECO ILN   (1961-2025)
  downloaded defl      <- AMECO PCPH  (1960-2027)
  downloaded hhdi_nom  <- AMECO UVGH  (1985-2027)

Final sample: 1960-2024, N = 65

1. DESCRIPTIVES, LINEAR TREND AND EXPONENTIAL GROWTH
Sample 1960-2024  (N = 65).  Variables available: ['lgdp', 'lcons', 'linv', 'lrhdi', 'rate', 'unemp', 'infl']
Average consumption/GDP ratio = 0.525  (first year 0.565, last 0.500)
variable  linear_R2  exp_R2  growth_cont_pct  growth_compound_pct
     gdp     0.9948  0.9636            2.309                2.336
    cons     0.9964  0.9526            2.220                2.245

2. UNIT ROOT TESTS   (ADF H0: unit root | KPSS H0: stationary | Zivot-Andrews H0: unit root, no break)
       series determin